# Confidence-Based Failure Detection for Liver Cancer Segmentator

This notebook evaluates whether ensemble agreement can identify unreliable nnUNet liver/tumor segmentations before clinical review.

The analysis reports Dice-based failure risk, pairwise ensemble confidence, risk-coverage curves, AURC, and cohort-level reliability summaries.

## Method

- Compute tumor Dice for the ensembled prediction.
- Convert Dice to failure risk with `risk = 1 - Dice`.
- Estimate confidence from mean pairwise Dice agreement across the five fold predictions.
- Evaluate ranking quality with AURC: lower values mean high-risk failures are rejected earlier as coverage decreases.
- Stratify results by cohort, tumor volume, and slice thickness.

In [ ]:
from pathlib import Path

import pandas as pd

from failureDetection.failure_detection import collect_case_metrics, summarize_reliability
from failureDetection.plots import plot_cohort_reliability, plot_risk_coverage

pd.set_option("display.max_columns", 50)

## Configuration

Use local paths for private data. Keep patient data, labels, and prediction masks out of Git.

In [ ]:
REFERENCE_DIR = Path("data/labels")
ENSEMBLE_DIR = Path("data/predictions/ensemble")
FOLD_DIRS = [Path(f"data/predictions/fold_{i}") for i in range(5)]
OUTPUT_DIR = Path("failureDetection/outputs")
LABEL_VALUE = 2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Collect Case-Level Metrics

In [ ]:
df = collect_case_metrics(
    reference_dir=REFERENCE_DIR,
    ensemble_dir=ENSEMBLE_DIR,
    fold_dirs=FOLD_DIRS,
    label_value=LABEL_VALUE,
)

df.head()

In [ ]:
valid_df = df[df["status"].eq("ok")].copy()
valid_df[["dice", "risk", "confidence_pairwise_dice", "tumor_volume_mm3", "slice_thickness_mm"]].describe()

## Risk-Coverage Evaluation

In [ ]:
summary_df = summarize_reliability(df)
summary_df.sort_values("aurc")

In [ ]:
fig, ax = plot_risk_coverage(valid_df)
fig.savefig(OUTPUT_DIR / "risk_coverage_curve.png", bbox_inches="tight")

In [ ]:
fig, ax = plot_cohort_reliability(summary_df)
fig.savefig(OUTPUT_DIR / "cohort_reliability.png", bbox_inches="tight")

## Metadata Checks

These lightweight checks help identify whether acquisition metadata or tumor burden is associated with low segmentation reliability.

In [ ]:
valid_df.groupby("cohort").agg(
    cases=("case_id", "count"),
    mean_dice=("dice", "mean"),
    mean_confidence=("confidence_pairwise_dice", "mean"),
    mean_risk=("risk", "mean"),
    median_tumor_volume_mm3=("tumor_volume_mm3", "median"),
    median_slice_thickness_mm=("slice_thickness_mm", "median"),
).sort_values("mean_risk", ascending=False)

## Takeaways

Use this section for final results before publishing the repository. Good public conclusions should be specific and quantitative, for example:

- Pairwise ensemble agreement achieved lower AURC than the baseline confidence score.
- Low-confidence cases were enriched for low Dice predictions.
- Reliability differed across cohorts and acquisition settings.
- Slice thickness and tumor burden were useful metadata for interpreting failure modes.